In [ ]:
# Cognitive Skills & Student Performance Analysis

This notebook analyzes the relationship between cognitive skills and student performance, builds ML models, and performs clustering analysis.

## Dataset Overview
- **200 students** across grades 9-12
- **Cognitive skills**: comprehension, attention, focus, retention
- **Performance metrics**: assessment_score, engagement_time


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
# Load the dataset
df = pd.read_csv('../data/student_data.csv')
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nBasic statistics:")
print(df.describe())


In [ ]:
## 1. Correlation Analysis


In [ ]:
# Calculate correlation matrix
cognitive_skills = ['comprehension', 'attention', 'focus', 'retention']
performance_metrics = ['assessment_score', 'engagement_time']
all_metrics = cognitive_skills + performance_metrics

correlation_matrix = df[all_metrics].corr()

# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: Cognitive Skills vs Performance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print correlation with assessment score
print("Correlation with Assessment Score:")
print(correlation_matrix['assessment_score'].sort_values(ascending=False))


In [ ]:
## 2. Machine Learning Model - Predicting Assessment Score


In [ ]:
# Prepare data for ML
X = df[cognitive_skills + ['engagement_time']]
y = df['assessment_score']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MSE': mse, 'R2': r2, 'predictions': y_pred}

# Display results
print("Model Performance:")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  MSE: {metrics['MSE']:.2f}")
    print(f"  R²: {metrics['R2']:.3f}")
    print()

# Feature importance for Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance)


In [ ]:
## 3. Student Clustering - Learning Personas


In [ ]:
# Prepare data for clustering
cluster_features = cognitive_skills + ['engagement_time']
X_cluster = df[cluster_features]

# Scale the features
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Determine optimal number of clusters using elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# Perform clustering with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Analyze clusters
cluster_analysis = df.groupby('cluster')[cluster_features + ['assessment_score']].mean()
print("Cluster Analysis:")
print(cluster_analysis.round(2))

# Define learning personas based on cluster characteristics
persona_names = {
    0: "High Performers",
    1: "Focused Learners", 
    2: "Engaged Students",
    3: "Developing Learners"
}

df['persona'] = df['cluster'].map(persona_names)

# Display persona distribution
print("\nPersona Distribution:")
print(df['persona'].value_counts())


In [ ]:
## 4. Visualization and Insights


In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Skill vs Score scatter plot
axes[0, 0].scatter(df['comprehension'], df['assessment_score'], alpha=0.6, c=df['cluster'], cmap='viridis')
axes[0, 0].set_xlabel('Comprehension')
axes[0, 0].set_ylabel('Assessment Score')
axes[0, 0].set_title('Comprehension vs Assessment Score')

# 2. Attention vs Performance
axes[0, 1].scatter(df['attention'], df['assessment_score'], alpha=0.6, c=df['cluster'], cmap='viridis')
axes[0, 1].set_xlabel('Attention')
axes[0, 1].set_ylabel('Assessment Score')
axes[0, 1].set_title('Attention vs Performance')

# 3. Persona distribution
persona_counts = df['persona'].value_counts()
axes[0, 2].pie(persona_counts.values, labels=persona_counts.index, autopct='%1.1f%%')
axes[0, 2].set_title('Learning Persona Distribution')

# 4. Average skills by persona
persona_skills = df.groupby('persona')[cognitive_skills].mean()
persona_skills.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Average Cognitive Skills by Persona')
axes[1, 0].set_ylabel('Score')
axes[1, 0].tick_params(axis='x', rotation=45)

# 5. Assessment score distribution by persona
df.boxplot(column='assessment_score', by='persona', ax=axes[1, 1])
axes[1, 1].set_title('Assessment Score Distribution by Persona')
axes[1, 1].set_xlabel('Persona')

# 6. Engagement time vs performance
axes[1, 2].scatter(df['engagement_time'], df['assessment_score'], alpha=0.6, c=df['cluster'], cmap='viridis')
axes[1, 2].set_xlabel('Engagement Time (minutes/week)')
axes[1, 2].set_ylabel('Assessment Score')
axes[1, 2].set_title('Engagement Time vs Performance')

plt.tight_layout()
plt.show()


In [ ]:
## 5. Key Insights and Recommendations


In [ ]:
# Generate insights
print("=== KEY INSIGHTS ===\n")

# 1. Overall performance statistics
print("1. OVERALL PERFORMANCE:")
print(f"   • Average Assessment Score: {df['assessment_score'].mean():.1f}")
print(f"   • Average Engagement Time: {df['engagement_time'].mean():.1f} minutes/week")
print(f"   • Students with Score > 80: {(df['assessment_score'] > 80).sum()} ({(df['assessment_score'] > 80).mean()*100:.1f}%)")

# 2. Correlation insights
print("\n2. STRONGEST CORRELATIONS WITH ASSESSMENT SCORE:")
correlations = correlation_matrix['assessment_score'].sort_values(ascending=False)
for skill, corr in correlations.items():
    if skill != 'assessment_score':
        print(f"   • {skill.title()}: {corr:.3f}")

# 3. Persona insights
print("\n3. LEARNING PERSONA INSIGHTS:")
for persona in df['persona'].unique():
    persona_data = df[df['persona'] == persona]
    avg_score = persona_data['assessment_score'].mean()
    count = len(persona_data)
    print(f"   • {persona}: {count} students, Avg Score: {avg_score:.1f}")

# 4. Model performance
print("\n4. ML MODEL PERFORMANCE:")
best_model = max(results.items(), key=lambda x: x[1]['R2'])
print(f"   • Best Model: {best_model[0]} (R² = {best_model[1]['R2']:.3f})")

# 5. Recommendations
print("\n5. RECOMMENDATIONS:")
print("   • Focus on comprehension and retention skills for better performance")
print("   • High engagement time correlates with better scores")
print("   • Different learning personas need tailored approaches")
print("   • Early intervention for 'Developing Learners' persona")

# Save processed data for dashboard
df.to_csv('../data/student_data_with_personas.csv', index=False)
print("\n✅ Processed data saved to '../data/student_data_with_personas.csv'")
